# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [ ]:
import pandas as pd
import numpy as np
import re

## Variable global
content = ""
sections_content = {}
data_report = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGU_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [ ]:
section_header_pattern        = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)
s1_kv_pattern                 = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)
s2_header_pattern             = re.compile(r"(Numéro)\s+(Début)\s+-\s+(Fin)\s+(Centroïde)\s+(Energie)\s+(FWHM)\s+(Surface)\s+(Incert\.)\s+(Fond sous)\s*\r?\n^\s*(du pic)\s+(\(canaux\))\s+(\(keV\))\s+(\(keV\))\s+(le pic)", re.MULTILINE)
s2_data_pattern               = re.compile(r"^\s*([MmF]?)\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+-\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)", re.MULTILINE)
s3_header_pattern             = re.compile(r"^\s+(Nom\sdu)\s+(Indice\sde)\s+(Energie)\s+(Intensité)\s+(Activité)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(\W\w+\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\n", re.MULTILINE)
s3_data_pattern               = re.compile(r"^\s*([A-Z]{1,2}-\d{1,3})?\s*(\d+\.\d+)?\s*(\d+\.\d+)(\*?)\s*(@?)\s*(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s6_header_pattern             = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
s6_word_column_pattern        = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_nucleide_line_pattern      = re.compile(r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)", re.MULTILINE)


## Extraction des header de chaque section
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [ ]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())
    data_report[section.strip()] = None

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport.

Pour l'instant ce n'est pas encore bon:
Mauvaise dections des clés/valeurs

In [ ]:
content_s1 = sections_content[sections_titles[0]]
matches = s1_kv_pattern.findall(content_s1)
data_report[sections_titles[0]] = {key.strip(): value.strip() for key, value in matches}

####### DATA ##########
data_report[sections_titles[0]]

### S2 - RAPPORT ANALYSE DES PICS

In [ ]:
def extract_header_s2(content):
    match = re.search(s2_header_pattern, content)
  
    if not match:
        return None
        
    columns = [
        "Marker",                                   # Marqueur (M, m, F)
        f"{match.group(1)} {match.group(9)}",       # Numéro du pic
        f"{match.group(2)} {match.group(10)}",        # Début (canaux)       
        f"{match.group(3)} {match.group(11)}",                       # Fin (canaux)
        match.group(4),                             # Centroïde
        f"{match.group(5)} {match.group(12)}",      # Energie (keV)
        f"{match.group(6)} {match.group(13)}",      # FWHM (keV)
        match.group(7),                             # Surface
        match.group(8),                             # Incert.
        f"{match.group(8)} {match.group(13)}"       # Fond sous le pic
    ]
    
    return columns



In [ ]:
def extract_data_s2(content, header):
    matches = re.findall(s2_data_pattern, content)
    data = pd.DataFrame(matches, columns=header)
    data[header[1:]] = data[header[1:]].apply(pd.to_numeric, errors="coerce")
    
    return data


In [ ]:
content_s2 = sections_content[sections_titles[1]]
header_s2 = extract_header_s2(content_s2)
data_report[sections_titles[1]] = extract_data_s2(content_s2, header_s2)

####### DATA ##########
data_report[sections_titles[1]]

### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

In [ ]:
def extract_header_s3(content):
    matches = re.search(s3_header_pattern, content)

    columns = [
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        "Marker *",
        "Marker @",
        f"{matches.group(4)} {matches.group(10)}",
        f"{matches.group(5)} {matches.group(11)}",
        f"{matches.group(6)} {matches.group(12)}"
    ]
   
    return columns

In [ ]:
def extract_data_s3(content, header):
    matches = re.findall(s3_data_pattern, content)
    df = pd.DataFrame(matches ,columns=header)
    
    numeric = ['Indice de confiance', 'Energie (keV)', 'Intensité (%)', 'Activité (mBq/g   )', 'Incert. (mBq/g   )']
    df[numeric] = df[numeric].apply(pd.to_numeric, errors="coerce") 
    
    return df

In [ ]:
content_s3 = sections_content[sections_titles[2]]
header_s3 = extract_header_s3(content_s3)
data_report[sections_titles[2]] = extract_data_s3(content_s3, header_s3)

### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

In [ ]:
content_s4 = sections_content[sections_titles[3]]

### S5 - RAPPORT LIMITES DE DETECTION

In [ ]:
content_s5 = sections_content[sections_titles[4]]

### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [ ]:
def extract_header_s6(content):
    header = []
    matches = re.findall(s6_header_pattern, content)

    l1 = re.findall(s6_word_column_pattern, matches[0][0])
    l2 = re.findall(s6_word_column_pattern, matches[0][1])
    
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header

In [ ]:
def extract_data_s6(content, header):
    matches = re.findall(s6_nucleide_line_pattern, content)
    data = pd.DataFrame(matches, columns=header)
    data[header[1:]] = data[header[1:]].apply(pd.to_numeric, errors="coerce") 
    
    return data

In [ ]:
content_s6 = sections_content[sections_titles[5]]
header = extract_header_s6(content_s6)
data_report[sections_titles[5]] = extract_data_s6(content_s6, header)

####### DATA ##########
data_report[sections_titles[5]]

# Zone de Test

In [ ]:
print(data_report)
